# 1 - IP Subset HUC8 Polygons by NetMap Extent


The intended use of this tool is to subset the National Hydrography Dataset (NHD) HUC 8 polygons that overlap a given NetMap synthetic streamline dataset. (NHD data can be downloaded here: https://www.usgs.gov/national-hydrography/access-national-hydrography-products)

## Required Software:

- The code contained in this notebook is designed to be run within an ESRI ArcPro project. The script output is written to the default project geodatabase and therefore the user must open and run the notebook .IPYNB file within an ArcPro project.
- If any of the geoprocessing steps require an advanced license or any specific extensions, the script will check for these conditions before running.


## Required Inputs:

- A geodatabase containing the unprocessed, high resolution NetMap synthetic stream reach dataset.

- The complete NHD Alaska HUC8 polygon layer.

- A prefix to use in naming the subset HUC8 feature class that will be saved in the geodatabase. It is recommended to use the same prefix as used in the NetMap dataset.


## Geoprocessing Output:

- A polygon feature class in the user-provided geodatabase, containing the HUC8 polygons. The feature class will have a suffix of "HUC8". (The suffix will allow further processing of the feature classes using only the geodatabase as input in other tools.) **This feature class will need to be visually QC'd to delete any HUC 8 polygons included as a result of very small NetMap reaches crossing the HUC 8 borders. The user may also need to manually delete the temporary "reach_" feature layer.**

## Processing Steps:

1. Define the user-provided geodatabase as the workspace environment.
2. Create a feature layer from the synthetic streamlines.
3. Select HUC8 polygons that contain any part of the streamline feature layer.
4. Export the selection as a feature class, adding a "HUC8" suffix.
5. Delete the streamline feature layer.

### Code starts here:

#### Setup

Import modules and set environment to the user-provided geodatabase. Additionally, allow  the addition of intermediary outputs to the ArcPro project map.

In [ ]:
import arcpy
arcpy.env.addOutputsToMap = True

User provides filepaths to the geodatabase, stream reaches, and NHD HUC8 polygons within the geodatabase. A unique dataset prefix is also provided.

(These inputs are provided directly as text when running the code block from the notebook. Use the hashed out code block below if saving the notebook as a .PY file and creating a tool for use in the ArcPro GUI. In that case, the inputs will be provided as text parameters in "point-and-click" fashion when setting up the tool in the ArcPro GUI.)

In [ ]:
work = "F:/GIS/IP/Yukon_Upper_2.gdb"
reach = "reach_Upper_Yukon2"
prefix = "Upper_Yukon2"

huc8 = "WBDHU8"
# when running from the notebook, the HUC 8 polygons can simply be loaded into the map
# and do not need to be copied to the project geodatabase

#work = arcpy.GetParameterAsText(0)
#reach = arcpy.GetParameterAsText(1)
#prefix = arcpy.GetParameterAsText(2)

Set the environment to match the workspace GDB that contains the NetMap dataset. Import the reaches as a feature layer.

In [ ]:
arcpy.env.workspace = work

arcpy.management.MakeFeatureLayer(reach, "reach_")

Select the HUC 8 polygons that contain NetMap reaches.

In [ ]:
arcpy.management.SelectLayerByLocation(huc8, "CONTAINS", "reach_", "", "NEW_SELECTION", "")

Export the selected polygons to a new feature class in the workspace GDB. This feature class will need to be visually QC'd to delete any HUC 8 polygons included as a result of very small NetMap reaches crossing the HUC 8 borders. Then delete the temporary "reach_" feature layer.

In [ ]:
huc_export = str(prefix + "HUC8")
arcpy.management.CopyFeatures(huc8, huc_export)

In [ ]:
try:
    arcpy.management.Delete("reach_")
except:
    desc = arcpy.Describe("reach_")
    arcpy.management.Delete(desc.path)